# 03 · Feature Engineering

Produces `dgstats_panel.parquet`, `caiso_daily.parquet`, and `caiso_monthly.parquet`.

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

PROC = ROOT / 'data' / 'processed'

import pandas as pd
from cleaning import aggregate_dgstats_panel
from feature_engineering import compute_monthly_ramp

## 3.1 DGStats ZIP × Year Panel

In [2]:
dgstats = pd.read_parquet(PROC / 'dgstats_clean.parquet')
panel = aggregate_dgstats_panel(dgstats)
print(f'DGStats panel: {len(panel):,} ZIP-year rows | {panel["zip_code"].nunique():,} unique ZIPs')
panel.head()

DGStats panel: 12,131 ZIP-year rows | 1,618 unique ZIPs


,zip_code,install_year,annual_capacity_kw,install_count,storage_paired_count,btm_capacity_kw
0,90001,2015,109.09698,23,0,109.09698
1,90001,2016,104.06200,21,0,213.15898
2,90001,2017,56.99000,14,0,270.14898
3,90001,2018,54.47000,13,0,324.61898
4,90001,2019,51.11000,11,0,375.72898


In [3]:
panel.to_parquet(PROC / 'dgstats_panel.parquet', index=False)

## 3.2 CAISO Monthly Outcomes

Reads the cleaned monthly CAISO data and adds `monthly_ramp_gwh` (month-over-month
absolute change in net load). These columns become the regression outcome variables:
- `monthly_ramp_gwh`: grid stress proxy — larger swings indicate harder ramping
- `net_load_ratio`: net load / gross demand — lower = more solar displacing load

In [4]:
caiso = pd.read_parquet(PROC / 'caiso_clean.parquet')
print(f'CAISO monthly rows: {len(caiso):,}')
print('Columns:', list(caiso.columns))
caiso.head(3)

CAISO monthly rows: 108
Columns: ['year', 'month', 'demand_gwh', 'solar_gwh', 'wind_gwh', 'total_generation_gwh', 'net_load_gwh', 'net_load_ratio', 'monthly_ramp_gwh']


,year,month,demand_gwh,solar_gwh,wind_gwh,total_generation_gwh,net_load_gwh,net_load_ratio,monthly_ramp_gwh
0,2015,1,17803.362,1253.833333,1015.0,16349.583333,15534.528667,0.872562,NaN
1,2015,2,15997.661,1253.833333,1015.0,16349.583333,13728.827667,0.858177,1805.701
2,2015,3,18211.601,1253.833333,1015.0,16349.583333,15942.767667,0.875418,2213.940


In [5]:
# Ensure monthly_ramp_gwh is present (recompute in case clean parquet predates it)
if 'monthly_ramp_gwh' not in caiso.columns:
    caiso = compute_monthly_ramp(caiso)

monthly = caiso[['year', 'month', 'demand_gwh', 'solar_gwh', 'wind_gwh',
                  'net_load_gwh', 'net_load_ratio', 'monthly_ramp_gwh']].copy()
print(f'CAISO monthly outcome rows: {len(monthly):,}')
monthly.head()

CAISO monthly outcome rows: 108


,year,month,demand_gwh,solar_gwh,wind_gwh,net_load_gwh,net_load_ratio,monthly_ramp_gwh
0,2015,1,17803.362,1253.833333,1015.0,15534.528667,0.872562,NaN
1,2015,2,15997.661,1253.833333,1015.0,13728.827667,0.858177,1805.701
2,2015,3,18211.601,1253.833333,1015.0,15942.767667,0.875418,2213.940
3,2015,4,17555.875,1253.833333,1015.0,15287.041667,0.870765,655.726
4,2015,5,18218.457,1253.833333,1015.0,15949.623667,0.875465,662.582


In [6]:
monthly.to_parquet(PROC / 'caiso_monthly.parquet', index=False)
print('Saved caiso_monthly.parquet')

Saved caiso_monthly.parquet


## 3.3 DeepSolar Tract → ZIP Aggregation

In [7]:
from pathlib import Path
RAW = ROOT / 'data' / 'raw'

deepsolar = pd.read_parquet(RAW / 'deepsolar_ca_tracts.parquet')
xwalk = pd.read_parquet(RAW / 'zip_tract_xwalk.parquet')

# Standardize FIPS to 11-char string
deepsolar['fips'] = deepsolar['fips'].astype(str).str.zfill(11)
xwalk['tract'] = xwalk['tract'].astype(str).str.zfill(11)

# Merge on tract FIPS
ds_zip = deepsolar.merge(xwalk, left_on='fips', right_on='tract', how='inner')

# Area-weighted aggregation (weight by res_ratio if available)
weight_col = 'res_ratio' if 'res_ratio' in ds_zip.columns else None
numeric_cols = [c for c in ['solar_system_count', 'solar_panel_area_divided_by_area'] if c in ds_zip.columns]

ds_agg = ds_zip.groupby('zip')[numeric_cols].sum().reset_index().rename(columns={'zip': 'zip_code'})
ds_agg['zip_code'] = ds_agg['zip_code'].astype(str).str.zfill(5)
print(f'DeepSolar → ZIP: {len(ds_agg):,} ZIPs')
ds_agg.to_parquet(PROC / 'deepsolar_zip.parquet', index=False)

DeepSolar → ZIP: 2,179 ZIPs
